# CNC Tool Wear Prediction

## Initial Data Inspection

Goal: understand dataset structure, variables, and identify the best target for predictive maintenance modeling.

In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv("../data/mill.csv")
df.head()

,Unnamed: 0,case,run,VB,time,DOC,feed,material,smcAC,smcDC,vib_table,vib_spindle,AE_table,AE_spindle
0,row_0,1,1,0.00,2,1.5,0.5,1,-0.017090,0.625000,0.078125,0.314941,0.087280,0.103760
1,row_1,1,2,NaN,4,1.5,0.5,1,0.307617,0.668945,0.075684,0.301514,0.086670,0.099487
2,row_2,1,3,NaN,6,1.5,0.5,1,-0.725098,0.913086,0.083008,0.295410,0.092773,0.104980
3,row_3,1,4,0.11,7,1.5,0.5,1,0.112305,0.131836,0.083008,0.316162,0.112915,0.139771
4,row_4,1,5,NaN,11,1.5,0.5,1,-0.122070,0.449219,0.107422,0.284424,0.095825,0.110474


In [8]:
df.shape

(167, 14)

In [9]:
df.columns



Index(['Unnamed: 0', 'case', 'run', 'VB', 'time', 'DOC', 'feed', 'material',
       'smcAC', 'smcDC', 'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
      dtype='object')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   167 non-null    object 
 1   case         167 non-null    int64  
 2   run          167 non-null    int64  
 3   VB           146 non-null    float64
 4   time         167 non-null    int64  
 5   DOC          167 non-null    float64
 6   feed         167 non-null    float64
 7   material     167 non-null    int64  
 8   smcAC        167 non-null    float64
 9   smcDC        167 non-null    float64
 10  vib_table    167 non-null    float64
 11  vib_spindle  167 non-null    float64
 12  AE_table     167 non-null    float64
 13  AE_spindle   167 non-null    float64
dtypes: float64(9), int64(4), object(1)
memory usage: 18.4+ KB


## Target variable Analysis (VB)
We inspect tool wear values, missing labels, and distribution for the first predictive modeling task.

In [11]:
df["VB"].describe()


count    146.000000
mean       0.337603
std        0.260528
min        0.000000
25%        0.150000
50%        0.285000
75%        0.467500
max        1.530000
Name: VB, dtype: float64

In [12]:
df["VB"].isnull().sum()

21

In [13]:
df_vb = df.dropna(subset = ["VB"]).copy()
df_vb.shape

(146, 14)

### Threshold Selection

VB = 0.30 was selected as the binary wear threshold because it lies near the median of observed wear values, creating a practical and relatively balanced classification target.

In [14]:
df_vb["VB"].describe()

count    146.000000
mean       0.337603
std        0.260528
min        0.000000
25%        0.150000
50%        0.285000
75%        0.467500
max        1.530000
Name: VB, dtype: float64

## Binary Target Creation

Converted wear values into healthy vs worn labels. Now this can be solved as a classification problem.

In [15]:
df_vb["wear_flag"] = (df_vb["VB"] >= 0.30).astype(int)
df_vb["wear_flag"].value_counts()

wear_flag
0    76
1    70
Name: count, dtype: int64

## Feature Selection and Trian Test Split
Balanced binary wear labels were predicted usig machine parameter and sensor variables. The original VB column was removed to prevent target leakage.

In [17]:
X = df_vb.drop(columns=["Unnamed: 0", "VB", "wear_flag"])
y = df_vb["wear_flag"]

X.columns
               

Index(['case', 'run', 'time', 'DOC', 'feed', 'material', 'smcAC', 'smcDC',
       'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
      dtype='object')

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
X_train.shape, X_test.shape

((116, 12), (30, 12))

In [23]:
X.columns

Index(['case', 'run', 'time', 'DOC', 'feed', 'material', 'smcAC', 'smcDC',
       'vib_table', 'vib_spindle', 'AE_table', 'AE_spindle'],
      dtype='object')

## Logistic Regression Baseline
Trained the first model to predict tool wear using the selected features.

In [24]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(max_iter = 1000)
lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [25]:
lr.score(X_test, y_test)

0.9333333333333333

The first model reached 93.3% accuracy on the test data

In [26]:
lr.coef_

array([[ 0.05475544,  0.15132929,  0.09098735,  1.66340623,  0.39332535,
         2.59124508,  0.21688188,  0.10357001, -0.10982007, -0.05504232,
         0.01613447, -0.01320751]])